# PA4 on Kaggle: run your agent and your team against Qwen3.5-9B

**Before you start:** Accelerator **GPU T4 x2**, Internet **On**. Put your repo where the "Get your code" cell can
find it (a private Kaggle dataset of your repo, or a git URL).

**Run long experiments as a commit:** *Save Version → Save & Run All (Commit)*. A committed run keeps going after you
close the tab (up to 12 h) and its files stay in the version's **Output** tab. Interactive sessions time out and
lose everything. Use interactive mode only to debug on one or two tasks.

What happens:
1. install the pinned stack and check your code with the offline smoke tests;
2. start vLLM serving Qwen3.5-9B on both T4s (**about 10 minutes, once per session**);
3. run the experiment configurations you list in `EXPERIMENTS`; every finished attempt is appended to
   `results.jsonl` immediately. Re-running the cell skips finished attempts and retries any that crashed;
4. summarise with your own `eval.py`, and zip results, transcripts and the server log for download.

**Continuing in a new commit:** a new commit starts with an empty output folder. Attach the earlier version's output
to this notebook as input and set `CONTINUE_FROM` to the path of its `results.jsonl`: finished attempts are skipped,
and the new `results.jsonl` holds every row, old and new, ready to submit.

**Rough GPU time for the 10 dev tasks × 3 repeats, 12 attempts at once:** `solo` about
45 min, `team` about 2 h, `solo_matched` about 1 h. The handout has the full table.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Versions that work together on Kaggle.
!pip install -q vllm==0.33.0 2>&1 | tail -n 3
!pip install -q openai==3.51.0 mcp==1.30.0 langgraph==1.2.22 langchain-core==1.8.0 pytest 2>&1 | tail -n 3

## Get the code

In [ ]:
# ---- Get your code -------------------------------------------------------------------------------------------
# Option A (recommended): upload your repo folder as a PRIVATE Kaggle dataset and attach it to this notebook.
# Option B: a git URL (a private repo needs a token; never commit a token into a notebook you share).
import shutil, subprocess
from pathlib import Path

DATASET_DIR = "/kaggle/input/datasets/YOUR-KAGGLE-USERNAME/YOUR-DATASET-NAME"   # Option A: shown in the Input panel
GIT_URL = ""                                  # Option B: e.g. "https://github.com/you/pa4.git"

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = WORK / "pa4repo"
if REPO.exists():
    shutil.rmtree(REPO)
if GIT_URL:
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
else:
    found = sorted(Path(DATASET_DIR).rglob("submission_adapter.py")) if Path(DATASET_DIR).is_dir() else []
    if not found:
        elsewhere = sorted(str(p.parent) for p in Path("/kaggle/input").rglob("submission_adapter.py"))
        raise FileNotFoundError(f"No submission_adapter.py under {DATASET_DIR}. Set DATASET_DIR to the path the "
                                f"Input panel shows for your dataset. Found your code here instead: {elsewhere}")
    shutil.copytree(found[0].parent, REPO)
print("repo at", REPO)

In [ ]:
# ---- Import your code and run the offline smoke tests -----------------------------------------------------------
import subprocess, sys
sys.path.insert(0, str(REPO))
smoke = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"], cwd=REPO,
                       capture_output=True, text=True)
print(smoke.stdout[-1500:])
if smoke.returncode != 0:
    print("WARNING: some smoke tests fail. Experiments with unfinished parts will crash; fix them offline first.")

from pa4 import eval as E
from pa4 import experiments as X
from pa4.llm import openai_llm_fn
from pa4.serve import start_server, stop_server
from pa4.tasks import list_tasks

import logging
for noisy in ("httpx", "mcp"):   # one line per model request and per MCP tool call would bury the results
    logging.getLogger(noisy).setLevel(logging.WARNING)

## Configuration

In [ ]:
# ---- What to run ------------------------------------------------------------------------------------------------
MODEL = "qwen3.5-9b"
TASK_ROOT = REPO / "tasks" / "dev"
TASKS = list_tasks(TASK_ROOT)            # all 10 dev tasks; use a subset while debugging, e.g. ["roman", "table"]
REPEATS = 3                              # 3 tries per task: needed for pass^3
CONCURRENCY = 12                         # attempts at once; many more can stall the server
# In this order. solo_matched must come after team (it spends the team's budget). See pa4/experiments.py.
EXPERIMENTS = ["solo", "team", "solo_matched"]
# Add for A2/A3/A4: "solo_mcp", "solo_naive_tools", "solo_noverify".  Add for B3: "team_deceptive", "team_deceptive_trust",
# "team_inject_open", "team_inject_gated", "team_inject_lp" (on FAULT_TASKS, FAULT_REPEATS).
FAULT_TASKS, FAULT_REPEATS = TASKS[:3], 1
AGENT_KWARGS = {"max_steps": 30}         # passed to run_agent for solo configs and best-of-N attempts
TEAM_CFG = {}                            # TeamConfig overrides, e.g. {"max_rounds": 6}
DEADLINE_H = 11.0                        # stop starting new configurations after this many hours
OUT = WORK / "pa4_out"
CONTINUE_FROM = ""
TRY_TASK = ""

## Model server

In [ ]:
# ---- Start the model server (about 10 minutes) --------------------------------------------------------------------
proc = start_server(MODEL, log_path=WORK / "vllm.log")
llm_fn = openai_llm_fn(model=MODEL)
probe = llm_fn([{"role": "user", "content": "Reply with the single word: ready"}], [])
print("model says:", probe["content"], probe["usage"])

## Try one task (interactive sessions)

Watch the real model work on a single task: every model reply, every tool call and every tool result, first for your
agent alone (Section A), then for your team (Section B). Use these cells in an **interactive** session: set
`TRY_TASK` in the configuration cell (for example `"roman"`), run the cells above once (the server stays up for the
whole session), then run these as often as you like. They do nothing while `TRY_TASK` is empty, so leave it empty for
a committed run.

After changing your code, get the new files into the session (re-run "Get your code" with your updated dataset),
then run the reload cell. Re-running the server cell reuses the server that is already up.

In [ ]:
# ---- Try one task: your agent alone ----
from pa4.agent import run_agent
from pa4.tasks import fresh_copy


def show_conversation(messages, width=400):
    for m in messages:
        if m["role"] == "system":
            continue
        for tc in m.get("tool_calls") or []:
            print(f"[{m['role']}] calls {tc['function']['name']} {tc['function']['arguments'][:width]}")
        if m.get("content"):
            print(f"[{m['role']}] {m['content'][:width]}")
        print()


if TRY_TASK:
    d = fresh_copy(TASK_ROOT, TRY_TASK, WORK / "try" / f"solo_{TRY_TASK}")
    r = run_agent(d, llm_fn, **AGENT_KWARGS)
    show_conversation(r["messages"])
    print({k: r[k] for k in ("solved", "stop", "steps", "tool_calls", "nudges", "prompt_tokens", "completion_tokens")})

In [ ]:
# ---- Try one task: your team ----
from pa4 import team as T
from pa4.tasks import fresh_copy
from pa4.team import TeamConfig, run_team

if TRY_TASK:
    d = fresh_copy(TASK_ROOT, TRY_TASK, WORK / "try" / f"team_{TRY_TASK}")
    recorder = X.RecordingLLM(llm_fn)
    rec = run_team(d, recorder, TeamConfig(**TEAM_CFG))
    names = {" ".join(p.split()[:4]): name for name, p in
             [("supervisor", T.SUPERVISOR_PROMPT), ("coder", T.CODER_PROMPT), ("tester", T.TESTER_PROMPT)]}
    for c in recorder.calls:
        who = names.get(c["role"], c["role"])
        what = "tool result" if c["input"]["role"] == "tool" else "message"
        print(f"[{who}] receives {what}: {c['input']['content'][:300]}")
        for tc in c["reply"]["tool_calls"]:
            print(f"[{who}] calls {tc['name']} {tc['arguments'][:300]}")
        if c["reply"]["content"]:
            print(f"[{who}] replies: {c['reply']['content'][:300]}")
        print()
    print({k: rec[k] for k in ("solved", "status", "rounds", "handoffs", "prompt_tokens", "completion_tokens")})

In [ ]:
# ---- After changing your code: reload it without restarting the server ----
import importlib
import pa4.agent, pa4.context, pa4.eval, pa4.experiments, pa4.mcp_server, pa4.team, pa4.tools

for module in (pa4.tools, pa4.context, pa4.agent, pa4.mcp_server, pa4.team, pa4.eval, pa4.experiments):
    importlib.reload(module)
print("reloaded your code from", REPO)

## Experiments

In [ ]:
# ---- Run the experiments ------------------------------------------------------------------------------------------
import json, shutil, time
OUT.mkdir(parents=True, exist_ok=True)
RESULTS = OUT / "results.jsonl"
if CONTINUE_FROM and not RESULTS.exists():
    shutil.copy(CONTINUE_FROM, RESULTS)
t_start = time.time()

def show(row):
    if "error" in row:
        print(f"  ERROR {row['job_id']}: {row['error'][:200]}", flush=True)
        return
    end = row.get("status") or row.get("stop")
    print(f"  {row['job_id']:<34} {'SOLVED' if row['solved'] else 'failed':<6} {str(end):<20} "
          f"tokens={row.get('prompt_tokens', 0) + row.get('completion_tokens', 0):>7} {row.get('wall_s')}s", flush=True)

rows = []
if RESULTS.exists():
    rows = [json.loads(l) for l in RESULTS.read_text(encoding="utf-8").splitlines() if l.strip()]
for config in EXPERIMENTS:
    if time.time() - t_start > DEADLINE_H * 3600:
        print(f"deadline reached; not starting {config}")
        continue
    is_fault = config.startswith(("team_deceptive", "team_inject"))
    tasks, reps = (FAULT_TASKS, FAULT_REPEATS) if is_fault else (TASKS, REPEATS)
    jobs = X.make_jobs(config, tasks, reps, TASK_ROOT, OUT, llm_fn, team_cfg=TEAM_CFG, agent_kwargs=AGENT_KWARGS,
                       team_rows=rows)
    print(f"=== {config}: {len(jobs)} jobs ({(time.time() - t_start) / 3600:.2f} h elapsed)", flush=True)
    rows = E.run_grid(jobs, RESULTS, concurrency=CONCURRENCY, on_result=show)
print(f"done in {(time.time() - t_start) / 3600:.2f} h")

## Summary

In [ ]:
# ---- Summarise (with your eval.py) and package the outputs --------------------------------------------------------
import json, shutil
rows = [json.loads(l) for l in RESULTS.read_text(encoding="utf-8").splitlines() if l.strip()]
summary = {"by_config": X.summarize(rows, k=REPEATS)}
PAIRS = [("team", "solo"), ("team", "solo_matched"), ("solo", "solo_naive_tools"), ("solo", "solo_noverify"),
         ("solo", "solo_mcp")]
for a, b in PAIRS:
    if any(r.get("config") == a for r in rows) and any(r.get("config") == b for r in rows):
        summary[f"{a}_vs_{b}"] = X.compare(rows, a, b)
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
try:
    import pandas as pd
    display(pd.DataFrame({c: {k: v for k, v in s.items() if k != "per_task"} for c, s in summary["by_config"].items()}).T)
    display(pd.DataFrame({c: s["per_task"] for c, s in summary["by_config"].items()}))
except Exception:
    print(json.dumps(summary, indent=2, default=str))
shutil.copy(WORK / "vllm.log", OUT / "vllm.log") if (WORK / "vllm.log").exists() else None
shutil.make_archive(str(WORK / "pa4_outputs"), "zip", OUT)
print("Download:", WORK / "pa4_outputs.zip")
if not TRY_TASK:   # in an interactive session, keep the server up for more tries
    stop_server(proc)